#  CatBoost Hyperparameter Tuning
This notebook fine-tunes the CatBoost model using Optuna for best performance on ESG dataset.

In [8]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import optuna
import joblib
from pathlib import Path
import numpy as np

print("Libraries imported successfully.")


Libraries imported successfully.


In [4]:
# === Load Preprocessed Data ===
DATA_PATH = Path("C:\ESG\data\company_esg_financial_dataset.csv")
TARGET = "ESG_Overall"

train_df = pd.read_csv("C:\ESG\\notebooks\data\esg_artifacts\\train_processed.csv")
test_df = pd.read_csv("C:\ESG\\notebooks\data\esg_artifacts\\test_processed.csv")

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


Train shape: (7000, 26), Test shape: (4000, 26)


In [5]:
# === Define Optuna Objective Function ===
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 1200),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5.0),
        "od_type": "Iter",
        "od_wait": 40,
        "loss_function": "RMSE",
        "verbose": 0,
        "random_seed": 42,
    }

    model = CatBoostRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
    return -scores.mean()


In [7]:
# === Run Hyperparameter Optimization ===
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

print("Best Parameters:", study.best_params)


[I 2025-10-06 02:18:34,195] A new study created in memory with name: no-name-34e33f2b-15cd-4f1c-848a-46a7a6028c8d
Best trial: 0. Best value: 0.594058:   5%|▌         | 1/20 [00:55<17:40, 55.84s/it]

[I 2025-10-06 02:19:30,029] Trial 0 finished with value: 0.59405761270738 and parameters: {'iterations': 452, 'depth': 8, 'learning_rate': 0.2822773395140306, 'l2_leaf_reg': 7.039507563811302, 'bagging_temperature': 0.9048778868112108, 'border_count': 210, 'random_strength': 2.1928663237049}. Best is trial 0 with value: 0.59405761270738.


Best trial: 1. Best value: 0.429137:  10%|█         | 2/20 [04:13<41:49, 139.43s/it]

[I 2025-10-06 02:22:47,977] Trial 1 finished with value: 0.4291371595871844 and parameters: {'iterations': 1063, 'depth': 9, 'learning_rate': 0.02117195637087047, 'l2_leaf_reg': 9.461771850999037, 'bagging_temperature': 0.8713030497983518, 'border_count': 220, 'random_strength': 2.08051950939492}. Best is trial 1 with value: 0.4291371595871844.


Best trial: 2. Best value: 0.406583:  15%|█▌        | 3/20 [04:28<23:26, 82.71s/it] 

...

[I 2025-10-06 02:32:17,005] Trial 17 finished with value: 0.2683939721427434 and parameters: {'iterations': 973, 'depth': 5, 'learning_rate': 0.18365530891141013, 'l2_leaf_reg': 4.562847302419747, 'bagging_temperature': 0.1405611560908035, 'border_count': 156, 'random_strength': 2.0497313673449735}. Best is trial 17 with value: 0.2683939721427434.


Best trial: 17. Best value: 0.268394:  95%|█████████▌| 19/20 [14:14<00:34, 34.76s/it]

[I 2025-10-06 02:32:48,949] Trial 18 finished with value: 0.31041005208850175 and parameters: {'iterations': 725, 'depth': 6, 'learning_rate': 0.1833484057061794, 'l2_leaf_reg': 4.405403768650818, 'bagging_temperature': 0.15116594412324807, 'border_count': 173, 'random_strength': 1.781593632621129}. Best is trial 17 with value: 0.2683939721427434.


Best trial: 17. Best value: 0.268394: 100%|██████████| 20/20 [15:32<00:00, 46.61s/it]

[I 2025-10-06 02:34:06,379] Trial 19 finished with value: 0.35221993196132706 and parameters: {'iterations': 978, 'depth': 7, 'learning_rate': 0.18948645028178784, 'l2_leaf_reg': 5.331502375847304, 'bagging_temperature': 0.1003928790475742, 'border_count': 174, 'random_strength': 1.4555419555654938}. Best is trial 17 with value: 0.2683939721427434.
Best Parameters: {'iterations': 973, 'depth': 5, 'learning_rate': 0.18365530891141013, 'l2_leaf_reg': 4.562847302419747, 'bagging_temperature': 0.1405611560908035, 'border_count': 156, 'random_strength': 2.0497313673449735}


In [10]:
# === Train Best Model and Evaluate ===
best_params = study.best_params
best_model = CatBoostRegressor(**best_params, random_seed=42)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("Test R2:", r2_score(y_test, y_pred))


0:	learn: 13.5709695	total: 17.7ms	remaining: 17.2s
1:	learn: 11.6493536	total: 23.5ms	remaining: 11.4s
2:	learn: 10.0119055	total: 37.2ms	remaining: 12s
3:	learn: 8.7081037	total: 42.5ms	remaining: 10.3s
4:	learn: 7.5073219	total: 49.2ms	remaining: 9.52s
5:	learn: 6.4998524	total: 54.3ms	remaining: 8.76s
6:	learn: 5.7225107	total: 59.5ms	remaining: 8.21s
7:	learn: 5.1029653	total: 66.6ms	remaining: 8.03s
8:	learn: 4.5320411	total: 72.3ms	remaining: 7.74s
9:	learn: 3.9957571	total: 78.4ms	remaining: 7.55s
10:	learn: 3.5246135	total: 84.9ms	remaining: 7.42s
11:	learn: 3.1827519	total: 91.3ms	remaining: 7.31s
12:	learn: 2.8389245	total: 96.9ms	remaining: 7.16s
13:	learn: 2.5744272	total: 104ms	remaining: 7.15s
14:	learn: 2.3407472	total: 115ms	remaining: 7.33s
15:	learn: 2.1461599	total: 120ms	remaining: 7.17s
16:	learn: 1.9753012	total: 129ms	remaining: 7.25s
17:	learn: 1.8227666	total: 134ms	remaining: 7.12s
18:	learn: 1.7129760	total: 140ms	remaining: 7.02s
19:	learn: 1.6159543	total:

In [13]:
# === Save the Tuned Model ===
OUT_DIR = Path("C:\ESG\\notebooks\data\esg_artifacts")
OUT_DIR.mkdir(exist_ok=True)

joblib.dump(best_model, OUT_DIR / "catboost_best_model.joblib")
print("Model saved successfully at:", OUT_DIR / "catboost_best_model.joblib")


Model saved successfully at: C:\ESG\notebooks\data\esg_artifacts\catboost_best_model.joblib
